In [56]:
import pandas as pd
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from langdetect import detect
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import re
from datasets import load_dataset

In [57]:
dataset=load_dataset("parquet",data_files="hv_dataset_tokenized.parquet")

In [58]:
df=dataset['train'].to_pandas()

In [17]:
df

,tokenized_text
0,"[antingen, stödjer, din, webbläsare, inte, jav..."


In [59]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\geeth\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [60]:
# Initialize stemmers for both languages
stemmer_en = SnowballStemmer("english")
stemmer_sv = SnowballStemmer("swedish")

In [61]:
# Stopwords for both languages
stop_words_en = set(stopwords.words('english'))
stop_words_sv = set(stopwords.words('swedish'))

In [62]:
def process_tokens(tokens):

    if len(tokens) == 0:  # Explicitly check for empty lists
        return ''
    
    # Detect the language of the whole tokenized text (joined back into a sentence)
    text = ' '.join(tokens)

    lang = detect(text)
    
    cleaned_tokens = []
    
    # Remove stopwords and apply stemming based on detected language
    if lang == 'en':
        cleaned_tokens = [stemmer_en.stem(token) for token in tokens if token not in stop_words_en]
    elif lang == 'sv':
        cleaned_tokens = [stemmer_sv.stem(token) for token in tokens if token not in stop_words_sv]
    
    return ' '.join(cleaned_tokens)

In [63]:
df['cleaned_text']=df['tokenized_text'].apply(process_tokens)

In [64]:
df['cleaned_text'].iloc[0][:2000]

'anting stödj webbläs javascript javascript inaktiver . webbplat funger bäst aktiver javascript . hopp huvudinnehållet sidhuvud student medarbet kontak sök personal bibliotek in english sök huvudmeny meny stäng meny webbplatsenom webbplatsenintegritetspolicycooki kakorwebbpubliceringwebbpubliceringwebborganisationwebborganisationwebbredaktörpublicerarefunktion webborganisationenskrivregl publicerarefråg svar kring webbplatsenhantering dokumentpublicer dokument optimizely f.d episerverskap formulär episerverwebbguidetillgänglighetsredogörelsetillgänglighetsredogörelsetillgänglighetsredogör hvplaytillgänglighetsredogör canvasutbildningutbildningsök program kurserv intresser ? yhutbildningyhutbildningspecialist svetsautomation 400 yhpelkraftteknik 400 yhpsök högskolan västsök högskolan västså sök ossurval platsfördelningurval platsfördelningalternativt urval socionomprogrammetantagningsstatistikreservplaceradstudi karriärvägledningöverklagabehörighetbehörighetbasårstabellsärskild behör yr

In [66]:
df[['cleaned_text']].to_parquet("hv_dataset_cleaned.parquet")

In [35]:
print(df.shape)


(1, 2)


In [38]:
print(df['cleaned_text'].apply(len).value_counts())

7976792    1
Name: cleaned_text, dtype: int64


In [24]:
# Vectorization using TF-IDF
tfidf_vectorizer = TfidfVectorizer()
X = tfidf_vectorizer.fit_transform(df['cleaned_text'])

In [25]:
print(X.shape)

(1, 64483)


Fine-tuning pretarined model

In [40]:
from datasets import Dataset

In [42]:
dataset = Dataset.from_pandas(df[['cleaned_text']])
dataset = dataset.rename_column("cleaned_text", "text")

In [43]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"  
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

c:\Users\geeth\anaconda3\lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\geeth\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

c:\Users\geeth\anaconda3\lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [45]:
# Check if pad_token is set
if tokenizer.pad_token is None:
    # Option 1: Set eos_token as pad_token
    tokenizer.pad_token = tokenizer.eos_token

In [46]:
def tokenize_function(examples):
    tokenized_dataset = dataset.map(lambda examples: tokenizer(
    examples['text'], 
    padding='max_length', 
    truncation=True
    ), batched=True)

    return tokenized_dataset

In [52]:
from transformers import TrainingArguments

In [54]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>={ACCELERATE_MIN_VERSION}'`

In [2]:
import torch

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
device

'cpu'

In [1]:
import pandas as pd

In [6]:
df=pd.read_parquet('hv_dataset_cleaned2.parquet')

In [7]:
df_text = df.astype(str).apply(lambda x: ' '.join(x), axis=1)

In [8]:
df_text

0    ['n' 'n' 'g' ... '5' ' ' 'p']
dtype: object

In [5]:
df_text.to_csv('hv_dataset_unique.txt', index=False, header=False)

In [1]:
import re
import pandas as pd

In [2]:
with open('cleaned_hvdata.txt', 'r', encoding='utf-8') as file:
    data = file.read()

In [15]:
allowed_single_words = [
    'email', 'contact','topics','english requirements','requirements','tuition'
]

In [25]:
irrelavant=['build number','host name','updated','chevron_right','expand_more','follow us','change password','click the link','choose language','facebook','header','search staff','search',
 'header',
 'search staff','news archive']

In [3]:
from langdetect import detect, DetectorFactory
# Setting a seed for reproducibility
DetectorFactory.seed = 0

In [66]:
text = "bibliotek"
language = detect(text)


In [ ]:


def pre_process(text):
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    
    # Remove punctuation
    text = re.sub(r'[^\w\s@.?]', '', text)  # Allow digits to remain (e.g., "2017")
    
    # Convert to lowercase
    text = text.lower()
    
    # Split the text into sentences
    sentences = text.split('\n')
    
    # Define allowed single words that should be retained
    allowed_single_words = [
    'email', 'contact','topics','english requirements','requirements','tuition'
    ]  # Set for faster lookup
    cleaned_sentences = []
    
    for sentence in sentences:
        # Remove leading/trailing whitespace from the sentence
        sentence = sentence.strip()
        
        # Skip sentences that are less than 2 characters long
        if len(sentence)==2:
            continue  # Skip this sentence if it's irrelevant
        # Check if the sentence is English
        try:
            if detect(sentence) != 'en':
                continue  # Skip non-English sentences
        except:
            continue         
        
        # Check if any irrelevant phrases are in the sentence
        if any(phrase in sentence for phrase in irrelavant):
            continue  # Skip this sentence if it contains any irrelevant phrases
        
        # If the sentence is valid, append it to the cleaned list
        if sentence!='':
         cleaned_sentences.append(sentence)

    # Join cleaned sentences back into a single string
    cleaned_text = '\n '.join(cleaned_sentences).strip()  # Remove unwanted whitespace
    return cleaned_text


In [5]:
processed_text = pre_process(data)


In [6]:
with open('hv_english.txt', 'w', encoding='utf-8') as f:
    f.write(processed_text)

In [24]:
with open("hv_english2.txt", "r") as file:
    text = file.read()

In [25]:
def remove_duplicates(text):
    # Split text into sentences/lines
    sentences = text.split('\n')
    
    # Create a set to track unique sentences
    seen = set()
    unique_sentences = []
    
    for sentence in sentences:
        # Add sentence to the list if it's not a duplicate
        if sentence not in seen:
            unique_sentences.append(sentence)
            seen.add(sentence)
    
    # Join the unique sentences back into a single string
    cleaned_text = '\n'.join(unique_sentences)
    return cleaned_text

In [26]:
cleaned_text = remove_duplicates(text)

In [27]:
with open("hv_english3.txt", "w") as file:
    file.write(cleaned_text)